# 03 - Build Indexes (FAISS + BM25)

**Tujuan:** membangun dua indeks yang dipakai kondisi B (dense) dan C
(hybrid + rerank) dari `data/processed/chunks.jsonl` (lihat §6
`perancangan.md`).

Langkah: (1) encode seluruh chunk dengan model embedding (fp16, GPU) ->
matriks embedding; (2) bangun indeks FAISS `IndexFlatIP` + normalisasi L2
(bukan IVF/HNSW - korpus ini terlalu kecil untuk itu, hanya menambah
variabel pengganggu); (3) bangun indeks BM25 dengan tokenizer Bahasa
Indonesia (Sastrawi: stopword removal + stemming); (4) sanity check kueri
contoh berdampingan; (5) catat waktu & ukuran indeks ke
`results/indexing_stats.json` (bagian dari analisis biaya, rumusan
masalah 4).

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")

In [2]:
import json
import time
from pathlib import Path

from config import load_config, set_seed

cfg = load_config("../configs/base.yaml")
set_seed(cfg.seed)

cfg.models.embedding, cfg.models.reranker

('BAAI/bge-m3', 'BAAI/bge-reranker-v2-m3')

## Muat chunks

In [3]:
chunks = [json.loads(line) for line in open("../data/processed/chunks.jsonl", encoding="utf-8")]
chunk_ids = [c["chunk_id"] for c in chunks]

assert len(chunk_ids) == len(set(chunk_ids)), "chunk_id tidak unik"
print(f"{len(chunks)} chunk dimuat dari data/processed/chunks.jsonl")

114 chunk dimuat dari data/processed/chunks.jsonl


## Encode dense embeddings

Model embedding (`cfg.models.embedding`, lihat `configs/base.yaml`) dimuat
fp16 di GPU dan dijalankan berbatch lewat `indexing.encode_chunks`.

In [4]:
import indexing

t0 = time.perf_counter()
embeddings = indexing.encode_chunks(
    chunks,
    model_name=cfg.models.embedding,
    batch_size=32,
    device="cuda",
)
t_encode = time.perf_counter() - t0

print(f"embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")
print(f"waktu encode: {t_encode:.1f} detik ({len(chunks) / t_encode:.1f} chunk/detik)")

D:\application\skripsi-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14899.63it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:  25%|██▌       | 1/4 [00:01<00:05,  1.71s/it]

Batches:  50%|█████     | 2/4 [00:02<00:02,  1.10s/it]

Batches:  75%|███████▌  | 3/4 [00:02<00:00,  1.18it/s]

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.73it/s]

Batches: 100%|██████████| 4/4 [00:03<00:00,  1.29it/s]

embeddings shape: (114, 1024), dtype: float32
waktu encode: 39.9 detik (2.9 chunk/detik)


## Bangun indeks FAISS (dense)

`IndexFlatIP` + normalisasi L2 (setara cosine similarity) - cukup untuk
skala ribuan chunk.

In [5]:
faiss_dir = Path("../indexes/faiss")

t0 = time.perf_counter()
indexing.build_faiss_index(embeddings, chunk_ids, faiss_dir)
t_faiss_build = time.perf_counter() - t0

faiss_size_bytes = sum(f.stat().st_size for f in faiss_dir.glob("*") if f.is_file())
print(f"indeks FAISS ditulis ke {faiss_dir} ({faiss_size_bytes / 1024:.1f} KB) dalam {t_faiss_build:.2f} detik")

indeks FAISS ditulis ke ..\indexes\faiss (464.4 KB) dalam 0.72 detik


## Bangun indeks BM25 (sparse)

Tokenisasi Bahasa Indonesia (Sastrawi: stopword removal lalu stemming, lihat
`indexing.tokenize_id`). Tokenizer yang sama dipakai ulang saat query lewat
`load_bm25_index` sehingga skor BM25 tetap konsisten.

In [6]:
bm25_dir = Path("../indexes/bm25")

t0 = time.perf_counter()
indexing.build_bm25_index(chunks, bm25_dir)
t_bm25_build = time.perf_counter() - t0

bm25_size_bytes = sum(f.stat().st_size for f in bm25_dir.glob("*") if f.is_file())
print(f"indeks BM25 ditulis ke {bm25_dir} ({bm25_size_bytes / 1024:.1f} KB) dalam {t_bm25_build:.2f} detik")

indeks BM25 ditulis ke ..\indexes\bm25 (116.7 KB) dalam 77.37 detik


## Sanity check: kueri contoh

Bandingkan top-5 hasil jalur dense vs BM25 secara berdampingan untuk satu
kueri contoh terkait topik korpus (panduan pengabdian masyarakat).

In [7]:
import faiss as faiss_lib
import numpy as np

sample_query = "syarat pengusul proposal pengabdian masyarakat"

faiss_index, faiss_ids = indexing.load_faiss_index(faiss_dir)
bm25_data = indexing.load_bm25_index(bm25_dir)

q_emb = indexing.encode_chunks(
    [{"text": sample_query}], model_name=cfg.models.embedding, batch_size=1, device="cuda"
)
faiss_lib.normalize_L2(q_emb)
D, I = faiss_index.search(q_emb, 5)
dense_hits = [(faiss_ids[i], float(score)) for i, score in zip(I[0], D[0])]

bm25_scores = bm25_data["bm25"].get_scores(indexing.tokenize_id(sample_query))
top_bm25_idx = np.argsort(bm25_scores)[::-1][:5]
bm25_hits = [(bm25_data["chunk_ids"][i], float(bm25_scores[i])) for i in top_bm25_idx]

chunks_by_id = {c["chunk_id"]: c for c in chunks}

print(f"Kueri: {sample_query!r}\n")
print("--- Top-5 Dense (FAISS) ---")
for cid, score in dense_hits:
    print(f"{score:.4f}  {cid}")
    print(f"    {chunks_by_id[cid]['text'][:120]!r}")

print("\n--- Top-5 BM25 ---")
for cid, score in bm25_hits:
    print(f"{score:.4f}  {cid}")
    print(f"    {chunks_by_id[cid]['text'][:120]!r}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15515.06it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.18it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

Kueri: 'syarat pengusul proposal pengabdian masyarakat'

--- Top-5 Dense (FAISS) ---
0.6424  lpmb-2026-panduan-usulan-pengmas-internal-universitas-airlan::p47::c001
    'Usulan proposal pengabdian kepada masyarakat diusulkan secara daring (online) melalui Cybercampus V2 menu Pengabdian Mas'
0.6409  lpmb-2026-panduan-usulan-pengmas-internal-universitas-airlan::p15::c002
    'kepada fakultas di lingkungan UNAIR dengan dilampiri Buku Panduan Pengabdian kepada Masyarakat. Fakultas menginformasika'
0.6282  lpmb-2026-panduan-usulan-pengmas-internal-universitas-airlan::p41::c002
    'Jenis kegiatan pengabdian kepada masyarakat mandiri sesuai bidang ilmu pengusul yang dapat diajukan adalah:\n\natau dimanf'
0.6147  lpmb-2026-panduan-usulan-pengmas-internal-universitas-airlan::p15::c001
    'yang tercantum dalam proposal atau Surat Keputusan Rektor. y. Seluruh pendanaan skema pengabdian kepada masyarakat dilak'
0.6130  lpmb-2026-panduan-usulan-pengmas-internal-universitas-airlan::p13::c001
    '

## ASSERT

In [8]:
assert embeddings.shape[0] == len(chunks), "jumlah baris embedding != jumlah chunk"
assert faiss_index.ntotal == len(chunks), "jumlah vektor FAISS != jumlah chunk"
assert faiss_ids == chunk_ids, "pemetaan posisi->chunk_id FAISS tidak sejajar urutan chunks.jsonl"
assert bm25_data["chunk_ids"] == chunk_ids, "pemetaan posisi->chunk_id BM25 tidak sejajar urutan chunks.jsonl"
assert len(dense_hits) == 5 and len(bm25_hits) == 5, "sanity query tidak mengembalikan 5 hasil"
print("Semua assert lolos.")

Semua assert lolos.


## Tulis `results/indexing_stats.json`

In [9]:
stats = {
    "n_chunks": len(chunks),
    "embedding_model": cfg.models.embedding,
    "embedding_dim": int(embeddings.shape[1]),
    "encode_seconds": round(t_encode, 2),
    "encode_chunks_per_second": round(len(chunks) / t_encode, 2),
    "faiss_build_seconds": round(t_faiss_build, 2),
    "faiss_index_bytes": faiss_size_bytes,
    "bm25_build_seconds": round(t_bm25_build, 2),
    "bm25_index_bytes": bm25_size_bytes,
}

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)
stats_path = results_dir / "indexing_stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"Ditulis {stats_path}")
stats

Ditulis ..\results\indexing_stats.json


{'n_chunks': 114,
 'embedding_model': 'BAAI/bge-m3',
 'embedding_dim': 1024,
 'encode_seconds': 39.92,
 'encode_chunks_per_second': 2.86,
 'faiss_build_seconds': 0.72,
 'faiss_index_bytes': 475539,
 'bm25_build_seconds': 77.37,
 'bm25_index_bytes': 119468}

## Ringkasan

In [10]:
print(f"Chunk terindeks: {len(chunks)}")
print(f"Dim embedding: {embeddings.shape[1]} ({cfg.models.embedding})")
print(f"FAISS: {faiss_size_bytes / 1024:.1f} KB, dibangun dalam {t_faiss_build:.2f}s")
print(f"BM25: {bm25_size_bytes / 1024:.1f} KB, dibangun dalam {t_bm25_build:.2f}s")
print(f"Total waktu (encode + build): {t_encode + t_faiss_build + t_bm25_build:.1f}s")

Chunk terindeks: 114
Dim embedding: 1024 (BAAI/bge-m3)
FAISS: 464.4 KB, dibangun dalam 0.72s
BM25: 116.7 KB, dibangun dalam 77.37s
Total waktu (encode + build): 118.0s
